# 01 — Validação das fontes externas

Checagens locais (sobre `data/raw/`, sem custo de BigQuery) antes da construção das features:

1. A `taxa_alfabetizacao` do Censo 2022 corresponde à população de 15 anos ou mais?
2. Os tipos de esgotamento sanitário têm sobreposição (categoria agregada)?
3. FUNDEB/SIOPE: em qual bimestre está o consolidado anual e qual coluna de valor cada indicador usa?
4. Cobertura de cada fonte sobre os 5.232 municípios da base de modelagem.

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)

RAW = Path("data/raw")

gold = pd.read_parquet(RAW / "gold_base.parquet")
censo = pd.read_parquet(RAW / "censo2022_municipio.parquet")
alf = pd.read_parquet(RAW / "censo2022_alfabetizacao.parquet")
esg = pd.read_parquet(RAW / "censo2022_esgotamento.parquet")
fundeb = pd.read_parquet(RAW / "fundeb_2023.parquet")
alunos = pd.read_parquet(RAW / "silver_alunos.parquet")
evol = pd.read_parquet(RAW / "gold_evolucao.parquet")

ids_gold = set(gold["id_municipio"].astype(str))
print("Municípios na base de modelagem:", len(ids_gold))

Municípios na base de modelagem: 5232


## 1. Taxa de alfabetização do Censo 2022 = população de 15 anos ou mais?

A tabela detalhada só contém faixas a partir de 15 anos. Se a taxa recalculada a partir dela coincidir com a `taxa_alfabetizacao` da tabela `municipio`, a definição está confirmada.

In [2]:
contagem = alf.pivot_table(
    index="id_municipio", columns="alfabetizacao", values="populacao", aggfunc="sum"
)
taxa_15_calc = 100 * contagem["Alfabetizadas"] / contagem.sum(axis=1)

comp = (
    censo.set_index("id_municipio")[["taxa_alfabetizacao"]]
    .join(taxa_15_calc.rename("taxa_15_calc"))
)
# A tabela `municipio` armazena a taxa na escala 0–1; a recalculada está em 0–100.
comp["taxa_oficial_pct"] = 100 * comp["taxa_alfabetizacao"]
comp["diferenca"] = comp["taxa_oficial_pct"] - comp["taxa_15_calc"]

print(comp.head().to_string())
print("\nDiferença (oficial em % - recalculada), mesma escala:")
print(comp["diferenca"].describe().to_string())

              taxa_alfabetizacao  taxa_15_calc  taxa_oficial_pct  diferenca
id_municipio                                                               
1400704                  0.87529      87.52867            87.529    0.00033
1200435                  0.77397     77.396543            77.397   0.000457
1400407                  0.91823     91.823247            91.823  -0.000247
1200328                  0.81876     81.875698            81.876   0.000302
1303908                  0.85195     85.194818            85.195   0.000182

Diferença (oficial em % - recalculada), mesma escala:
count      5570.0
mean    -0.000006
std       0.00029
min     -0.000499
25%     -0.000259
50%     -0.000009
75%      0.000243
max        0.0005


## 2. Esgotamento sanitário: hierarquia entre os tipos?

Hipótese: `Rede geral, rede pluvial ou fossa ligada à rede` = `Rede geral ou pluvial` + `Fossa séptica ou fossa filtro ligada à rede`. Se confirmada, somar os 9 tipos conta parte da população duas vezes.

In [3]:
esg_w = esg.pivot_table(
    index="id_municipio", columns="tipo_esgotamento_sanitario", values="populacao", aggfunc="sum"
).fillna(0)

AGREGADO = "Rede geral, rede pluvial ou fossa ligada à rede"
COMPONENTES = ["Rede geral ou pluvial", "Fossa séptica ou fossa filtro ligada à rede"]

print("Totais nacionais por tipo:")
print(esg_w.sum().sort_values(ascending=False).to_string())

dif_hierarquia = esg_w[AGREGADO] - esg_w[COMPONENTES].sum(axis=1)
print("\nAgregado - soma dos componentes (por município):")
print(dif_hierarquia.describe().to_string())

pop_esg_sem_agregado = esg_w.drop(columns=AGREGADO).sum(axis=1)
razao_pop = pop_esg_sem_agregado / censo.set_index("id_municipio")["populacao"]
print("\nPopulação coberta pela tabela de esgotamento / população total do Censo:")
print(razao_pop.describe().to_string())

Totais nacionais por tipo:
tipo_esgotamento_sanitario
Rede geral, rede pluvial ou fossa ligada à rede    126320738
Rede geral ou pluvial                              117771612
Fossa rudimentar ou buraco                          39290587
Fossa séptica ou fossa filtro não ligada à rede     26726073
Fossa séptica ou fossa filtro ligada à rede          8549126
Rio, lago, córrego ou mar                            4072542
Vala                                                 2980118
Outra forma                                          1503483
Não tinham banheiro nem sanitário                    1182315

Agregado - soma dos componentes (por município):
count    5570.0
mean        0.0
std         0.0
min         0.0
25%         0.0
50%         0.0
75%         0.0
max         0.0

População coberta pela tabela de esgotamento / população total do Censo:
count      5570.0
mean     0.999967
std      0.001929
min      0.872631
25%           1.0
50%           1.0
75%           1.0
max      1.066697


## 3. FUNDEB/SIOPE: bimestre do consolidado anual e coluna de valor por indicador

In [4]:
resumo_fundeb = (
    fundeb.groupby(["id_indicador", "bimestre"])
    .agg(
        municipios=("id_municipio", "nunique"),
        nulos_real=("valor_real", lambda s: s.isna().sum()),
        nulos_pct=("valor_percentual", lambda s: s.isna().sum()),
        mediana_real=("valor_real", "median"),
        mediana_pct=("valor_percentual", "median"),
    )
    .reset_index()
)
print(resumo_fundeb.to_string(index=False))

id_indicador  bimestre  municipios  nulos_real  nulos_pct  mediana_real  mediana_pct
          35         1        5563        5563          0           NaN        11.96
          35         2        5563        5563          0           NaN        19.25
          35         3        5564        5564          0           NaN        22.54
          35         4        5566        5566          0           NaN        24.55
          35         5        5565        5565          0           NaN        25.82
          35         6        5566        5566          0           NaN        23.94
          44         1        5566           0       5566      1150.215          NaN
          44         2        5566           0       5566      2551.665          NaN
          44         3        5566           0       5566      3990.270          NaN
          44         4        5566           0       5566      5459.690          NaN
          44         5        5566           0       5566      70

## 4. Cobertura das fontes sobre a base de modelagem

In [5]:
ids_por_fonte = {
    "censo2022_municipio": set(censo["id_municipio"]),
    "censo2022_alfabetizacao": set(alf["id_municipio"]),
    "censo2022_esgotamento": set(esg["id_municipio"]),
    "fundeb_2023": set(fundeb["id_municipio"]),
    "alunos_2024 (rede 3)": set(alunos.loc[alunos["ano"] == 2024, "id_municipio"].astype(str)),
    "indicador_2023": set(evol["id_municipio"].astype(str)),
}

for nome, ids in ids_por_fonte.items():
    print(f"{nome:<26} {len(ids_gold & ids):>5} de {len(ids_gold)}")

print("\nAlunos avaliados por município (rede municipal, 2024):")
print(alunos.loc[alunos["ano"] == 2024, "qtd_alunos"].describe().to_string())

censo2022_municipio         5232 de 5232
censo2022_alfabetizacao     5232 de 5232
censo2022_esgotamento       5232 de 5232
fundeb_2023                 5230 de 5232
alunos_2024 (rede 3)        5232 de 5232
indicador_2023              4611 de 5232

Alunos avaliados por município (rede municipal, 2024):
count         5452.0
mean      337.541636
std      1286.749357
min             10.0
25%             55.0
50%            114.0
75%            256.0
max          49820.0


## 5. Cobertura da Gold frente à Silver

A Silver tem resultados da rede municipal (rede 3) em 2024 para mais municípios do que a Gold. Quais ficaram de fora, de quais UFs, e de que porte?

In [6]:
UF_POR_CODIGO = {
    "11": "RO", "12": "AC", "13": "AM", "14": "RR", "15": "PA", "16": "AP", "17": "TO",
    "21": "MA", "22": "PI", "23": "CE", "24": "RN", "25": "PB", "26": "PE", "27": "AL",
    "28": "SE", "29": "BA", "31": "MG", "32": "ES", "33": "RJ", "35": "SP", "41": "PR",
    "42": "SC", "43": "RS", "50": "MS", "51": "MT", "52": "GO", "53": "DF",
}

silver_2024 = alunos[alunos["ano"] == 2024].copy()
silver_2024["id"] = silver_2024["id_municipio"].astype(str).str.zfill(7)
silver_2024["uf"] = silver_2024["id"].str[:2].map(UF_POR_CODIGO)
silver_2024["na_gold"] = silver_2024["id"].isin(ids_gold)

cobertura = silver_2024.groupby("uf").agg(
    municipios_silver=("id", "size"),
    fora_da_gold=("na_gold", lambda s: (~s).sum()),
    alunos_fora=("qtd_alunos", lambda s: s[~silver_2024.loc[s.index, "na_gold"]].sum()),
)
cobertura["pct_fora"] = (100 * cobertura["fora_da_gold"] / cobertura["municipios_silver"]).round(1)
print(cobertura.sort_values("fora_da_gold", ascending=False).to_string())

fora = silver_2024[~silver_2024["na_gold"]]
print(f"\nTotal fora da Gold: {len(fora)} municípios, {int(fora['qtd_alunos'].sum()):,} alunos")
print("\nPorte (alunos avaliados) — fora da Gold vs. na Gold:")
print(pd.DataFrame({
    "fora": fora["qtd_alunos"].describe(),
    "na_gold": silver_2024.loc[silver_2024["na_gold"], "qtd_alunos"].describe(),
}).round(1).to_string())

    municipios_silver  fora_da_gold  alunos_fora  pct_fora
uf                                                        
SC                291            53        25906      18.2
RS                456            35         8585       7.7
MG                830            29         1277       3.5
RN                167            28         6454      16.8
AC                 22            22         7098     100.0
AM                 62            14         4440      22.6
BA                407            13         1484       3.2
PA                144             8         4930       5.6
SP                626             5         2627       0.8
PR                399             4          225       1.0
GO                245             2           35       0.8
RJ                 92             2          228       2.2
TO                139             2          170       1.4
MA                217             1          309       0.5
MT                140             1           26       0

## 6. Indicadores do INEP: qual rede foi usada no enriquecimento anterior, e cobertura da nova fonte

A tabela `br_inep_indicadores_educacionais.municipio` traz os indicadores de 2023 por rede (Municipal, Pública, Total). Comparando com a Gold nos municípios em comum, identificamos a rede usada no enriquecimento anterior. Em seguida, verificamos se a rede municipal cobre os municípios que ficaram fora da Gold.


In [7]:
from src.data.inep import COLUNAS_INEP

inep = pd.read_parquet(RAW / "inep_indicadores_2023.parquet")
inep["id_municipio"] = inep["id_municipio"].astype(str).str.zfill(7)

# Coluna da Gold (enriquecimento anterior) -> coluna equivalente na Base dos Dados
COMPARACAO = {
    "mat_media_de_alunos_por_turma_etapas_de_ensino_ensino_fundamental_anos_iniciais_fun_ai_cat_0":
        "atu_ef_anos_iniciais",
    "had_ensino_fundamental_anos_iniciais_fun_ai_cat_0":
        "had_ef_anos_iniciais",
    "afd_percentual_de_docentes1_2_por_grupo4_de_adequacao_da_formacao_a_disciplina_que_leciona_"
    "e_etapa_modalidade_de_ensino_ensino_fundamental_anos_iniciais_grupo_1_fun_ai_cat_1":
        "afd_ef_anos_iniciais_grupo_1",
    "ird_percentual_de_escolas_segundo_faixa_do_indicador_de_regularidade_do_docente_ird_1_2_"
    "alta_4_5_edu_bas_cat_4":
        "ird_alta",
    "tdi_ensino_fundamental_anos_iniciais_fun_ai_cat_0":
        "tdi_ef_anos_iniciais",
    "tre_taxa_de_aprovacao_ensino_fundamental_de_8_e_9_anos_anos_iniciais_1_cat_fun_ai":
        "taxa_aprovacao_ef_anos_iniciais",
    "dsu_percentual_de_funcoes_docentes_com_curso_superior_por_etapa_modalidade_de_ensino_"
    "ensino_fundamental_anos_iniciais_fun_ai_cat_0":
        "dsu_ef_anos_iniciais",
}

g = gold.assign(id_municipio=gold["id_municipio"].astype(str).str.zfill(7))
g = g[["id_municipio", *COMPARACAO]]

linhas = []
for rede, grupo in inep.groupby("rede"):
    pares = g.merge(grupo, on="id_municipio", how="inner")
    for col_gold, col_bd in COMPARACAO.items():
        dif = (pares[col_gold].astype(float) - pares[col_bd].astype(float)).abs()
        validos = dif.notna().sum()
        linhas.append({
            "rede": rede,
            "indicador": col_bd,
            "pares_validos": int(validos),
            "pct_iguais": 100 * (dif < 0.01).sum() / validos,
            "dif_media_abs": dif.mean(),
        })

comparacao = pd.DataFrame(linhas)
print("% de municípios com valor igual (|diferença| < 0,01):")
print(comparacao.pivot(index="indicador", columns="rede", values="pct_iguais").round(1).to_string())
print("\nDiferença média absoluta:")
print(comparacao.pivot(index="indicador", columns="rede", values="dif_media_abs").round(3).to_string())

% de municípios com valor igual (|diferença| < 0,01):
rede                             Municipal  Pública  Total
indicador                                                 
afd_ef_anos_iniciais_grupo_1           0.0      1.0    1.0
atu_ef_anos_iniciais                   0.2      0.1    0.1
dsu_ef_anos_iniciais                   0.0      0.0    0.0
had_ef_anos_iniciais                   0.1      0.1    0.2
ird_alta                              42.8     42.2   42.1
taxa_aprovacao_ef_anos_iniciais        9.2      8.9    8.0
tdi_ef_anos_iniciais                   0.2      0.2    0.2

Diferença média absoluta:
rede                             Municipal  Pública   Total
indicador                                                  
afd_ef_anos_iniciais_grupo_1        21.051   18.488  18.486
atu_ef_anos_iniciais                15.804   15.805  15.426
dsu_ef_anos_iniciais                65.654   69.360  69.795
had_ef_anos_iniciais                 5.166    5.007   4.955
ird_alta                    

In [8]:
municipal = inep[inep["rede"] == "Municipal"]
ids_inep_municipal = set(municipal["id_municipio"])

s24 = alunos[alunos["ano"] == 2024].copy()
s24["id"] = s24["id_municipio"].astype(str).str.zfill(7)
ids_fora_gold = set(s24.loc[~s24["id"].isin(ids_gold), "id"])

print(f"Silver 2024 (rede municipal): {len(s24)} municípios")
print(f"  com alvo não nulo: {s24['indicador_alfabetizacao'].notna().sum()}")
print(f"  com indicadores INEP da rede municipal: {s24['id'].isin(ids_inep_municipal).sum()}")
print(f"Dos {len(ids_fora_gold)} fora da Gold, com indicadores INEP da rede municipal: "
      f"{len(ids_fora_gold & ids_inep_municipal)}")

nulos = (
    municipal[municipal["id_municipio"].isin(s24["id"])][list(COLUNAS_INEP)]
    .isna()
    .sum()
    .rename(index={col: nome for col, (nome, _) in COLUNAS_INEP.items()})
)
print("\nNulos por indicador (rede municipal, municípios da Silver 2024):")
print(nulos[nulos > 0].sort_values(ascending=False).to_string())

Silver 2024 (rede municipal): 5452 municípios
  com alvo não nulo: 5450
  com indicadores INEP da rede municipal: 5452
Dos 220 fora da Gold, com indicadores INEP da rede municipal: 220

Nulos por indicador (rede municipal, municípios da Silver 2024):
ce_adeq_docente_anos_iniciais_g5      5449
ce_adeq_docente_anos_iniciais_g4      5437
ce_docentes_superior_anos_iniciais    5376
ce_adeq_docente_anos_iniciais_g2      5376
ce_adeq_docente_anos_iniciais_g3      5375
ce_adeq_docente_anos_iniciais_g1      5372
ce_docentes_superior_pre_escola       5365
ce_horas_aula_creche                  5362
ce_horas_aula_pre_escola              5362
ce_horas_aula_infantil                5362
ce_horas_aula_anos_iniciais           2304
ce_horas_aula_2ano                    2304
ce_horas_aula_1ano                    2304
ce_docentes_superior_creche           1680
ce_distorcao_2ano                     1670
ce_docentes_superior_infantil         1668
ce_reprovacao_anos_iniciais           1646
ce_distorcao_1ano 

### 6.1 Consistência interna: qual fonte está íntegra?

Nenhuma rede da Base dos Dados reproduz a Gold, e a Base dos Dados tem um padrão de nulos implausível. Para identificar qual fonte tem colunas deslocadas, testamos identidades que dados corretos obedecem (grupos composicionais somam 100%) e comparamos medianas com faixas plausíveis.

In [9]:
from src.data.build_features import _mapa_censo_escolar

mapa_gold = {original: novo for original, (novo, _) in _mapa_censo_escolar().items()}
gold_ce = gold.rename(columns=mapa_gold)[list(mapa_gold.values())].astype(float)


def inep_por_rede(rede: str) -> pd.DataFrame:
    df = inep[inep["rede"] == rede]
    renomear = {col: nome for col, (nome, _) in COLUNAS_INEP.items()}
    return df.rename(columns=renomear)[list(renomear.values())].astype(float)


fontes = {
    "gold": gold_ce,
    "bd_municipal": inep_por_rede("Municipal"),
    "bd_total": inep_por_rede("Total"),
}

# 1) Medianas e % de nulos por feature, lado a lado
resumo = pd.concat(
    {f"mediana_{nome}": df.median() for nome, df in fontes.items()}
    | {f"pct_nulo_{nome}": 100 * df.isna().mean() for nome, df in fontes.items()},
    axis=1,
)
print(resumo.round(1).to_string())

# 2) Identidades composicionais: em dados corretos, cada grupo soma ~100%
grupos = {
    "adequação infantil (g1..g5)": [f"ce_adeq_docente_infantil_g{k}" for k in range(1, 6)],
    "adequação anos iniciais (g1..g5)": [f"ce_adeq_docente_anos_iniciais_g{k}" for k in range(1, 6)],
    "regularidade (4 faixas)": [
        "ce_regularidade_baixa", "ce_regularidade_media_baixa",
        "ce_regularidade_media_alta", "ce_regularidade_alta",
    ],
    "rendimento (aprov. + reprov. + aband.)": [
        "ce_aprovacao_anos_iniciais", "ce_reprovacao_anos_iniciais", "ce_abandono_anos_iniciais",
    ],
}

linhas = []
for nome_fonte, df in fontes.items():
    for nome_grupo, colunas in grupos.items():
        soma = df[colunas].sum(axis=1, min_count=len(colunas))  # só linhas completas
        completas = soma.notna().sum()
        linhas.append({
            "fonte": nome_fonte,
            "grupo": nome_grupo,
            "linhas_completas": int(completas),
            "pct_soma_100": 100 * ((soma - 100).abs() <= 1).sum() / completas if completas else float("nan"),
            "mediana_soma": soma.median(),
        })

identidades = pd.DataFrame(linhas)
print("\n% de linhas completas cuja soma fica entre 99 e 101:")
print(identidades.pivot(index="grupo", columns="fonte", values="pct_soma_100").round(1).to_string())
print("\nMediana da soma:")
print(identidades.pivot(index="grupo", columns="fonte", values="mediana_soma").round(1).to_string())

                                    mediana_gold  mediana_bd_municipal  mediana_bd_total  pct_nulo_gold  pct_nulo_bd_municipal  pct_nulo_bd_total
ce_alunos_turma_infantil                    16.4                  68.8              66.3            0.0                    0.0                0.0
ce_alunos_turma_creche                      15.4                   0.0               0.0            2.6                    0.0                0.0
ce_alunos_turma_pre_escola                  17.4                   9.1              10.3            0.0                    0.0                0.0
ce_alunos_turma_anos_iniciais               19.3                  10.0              12.5            0.0                    0.0                0.0
ce_alunos_turma_1ano                        18.3                   0.0               0.2            0.1                    0.2                0.0
ce_alunos_turma_2ano                        19.2                  18.8              20.4            0.1                    0



### 6.2 Planilhas oficiais do INEP: reprodução da Gold e cobertura

O leitor das planilhas oficiais extrai a linha localização Total / dependência Municipal — a mesma rede do alvo. Ela deve reproduzir os valores da Gold nos municípios em comum (o teste abaixo, sobre a planilha de alunos por turma, identifica empiricamente a linha usada no enriquecimento anterior). Em seguida, verificamos a cobertura dos municípios ausentes da Gold.

In [10]:
oficial = pd.read_parquet(RAW / "inep_oficial_2023.parquet")

g = gold_ce.copy()
g["id_municipio"] = gold["id_municipio"].astype(str).str.zfill(7)
pares = g.merge(oficial, on="id_municipio", suffixes=("_gold", "_oficial"), validate="one_to_one")

linhas = []
for feature in gold_ce.columns:
    a, b = pares[f"{feature}_gold"], pares[f"{feature}_oficial"]
    iguais = ((a - b).abs() < 0.01) | (a.isna() & b.isna())
    linhas.append({
        "feature": feature,
        "pct_iguais": 100 * iguais.mean(),
        "dif_max": (a - b).abs().max(),
    })
reproducao = pd.DataFrame(linhas).sort_values("pct_iguais")
print(f"Municípios comparados: {len(pares)}")
print(reproducao.round(3).to_string(index=False))

ids_oficial = set(oficial["id_municipio"])
print(f"\nDos {len(ids_fora_gold)} fora da Gold, com indicadores oficiais: {len(ids_fora_gold & ids_oficial)}")
print(f"Silver 2024 (rede municipal) com indicadores oficiais: {s24['id'].isin(ids_oficial).sum()} de {len(s24)}")

Municípios comparados: 5232
                           feature  pct_iguais  dif_max
          ce_alunos_turma_infantil       100.0      0.0
            ce_alunos_turma_creche       100.0      0.0
        ce_alunos_turma_pre_escola       100.0      0.0
     ce_alunos_turma_anos_iniciais       100.0      0.0
              ce_alunos_turma_1ano       100.0      0.0
              ce_alunos_turma_2ano       100.0      0.0
        ce_alunos_turma_multietapa       100.0      0.0
            ce_horas_aula_infantil       100.0      0.0
              ce_horas_aula_creche       100.0      0.0
          ce_horas_aula_pre_escola       100.0      0.0
       ce_horas_aula_anos_iniciais       100.0      0.0
                ce_horas_aula_1ano       100.0      0.0
                ce_horas_aula_2ano       100.0      0.0
       ce_adeq_docente_infantil_g1       100.0      0.0
  ce_adeq_docente_anos_iniciais_g1       100.0      0.0
       ce_adeq_docente_infantil_g2       100.0      0.0
  ce_adeq_docente_an

In [11]:
o = oficial.set_index("id_municipio")
print("Alta Floresta D'Oeste (1100015), alunos por turma — linha Total/Total:")
print(o.loc["1100015", [
    "ce_alunos_turma_infantil", "ce_alunos_turma_creche",
    "ce_alunos_turma_pre_escola", "ce_alunos_turma_anos_iniciais",
]].to_string())
print("Esperado (planilha bruta, inspeção): 18.2, 14.5, 20.0, 15.2")

print("\nIdentidades composicionais nos dados oficiais (% de linhas completas com soma entre 99 e 101):")
for nome_grupo, colunas in grupos.items():
    soma = oficial[colunas].sum(axis=1, min_count=len(colunas))
    completas = soma.notna().sum()
    print(f"  {nome_grupo:<42} {100 * ((soma - 100).abs() <= 1).sum() / completas:6.1f}%")

Alta Floresta D'Oeste (1100015), alunos por turma — linha Total/Total:
ce_alunos_turma_infantil         18.2
ce_alunos_turma_creche           14.5
ce_alunos_turma_pre_escola       20.0
ce_alunos_turma_anos_iniciais    20.4
Esperado (planilha bruta, inspeção): 18.2, 14.5, 20.0, 15.2

Identidades composicionais nos dados oficiais (% de linhas completas com soma entre 99 e 101):
  adequação infantil (g1..g5)                 100.0%
  adequação anos iniciais (g1..g5)            100.0%
  regularidade (4 faixas)                     100.0%
  rendimento (aprov. + reprov. + aband.)      100.0%


In [12]:
from src.data.inep_oficial import MAPA, _planilha

bruto = pd.read_excel(_planilha("atu"), sheet_name=0, header=None, dtype=object)
primeira_coluna = bruto.iloc[:, 0].astype(str).str.strip()
linha_codigos = primeira_coluna.index[primeira_coluna == "NU_ANO_CENSO"][0]

atu = bruto.iloc[linha_codigos + 1:].copy()
atu.columns = bruto.iloc[linha_codigos].astype(str).str.strip()
atu = atu[pd.to_numeric(atu["NU_ANO_CENSO"], errors="coerce") == 2023]
atu["id_municipio"] = pd.to_numeric(atu["CO_MUNICIPIO"]).astype("int64").astype(str).str.zfill(7)
atu["linha"] = (
    atu["NO_CATEGORIA"].astype(str).str.strip() + " / " + atu["NO_DEPENDENCIA"].astype(str).str.strip()
)
features_atu = list(MAPA["atu"].values())
for codigo, nome in MAPA["atu"].items():
    atu[nome] = pd.to_numeric(atu[codigo], errors="coerce")

candidatos = g[["id_municipio", *features_atu]].merge(
    atu[["id_municipio", "linha", *features_atu]], on="id_municipio", suffixes=("_gold", "_inep")
)
coincide = pd.concat(
    [
        ((candidatos[f"{f}_gold"] - candidatos[f"{f}_inep"]).abs() < 0.01)
        | (candidatos[f"{f}_gold"].isna() & candidatos[f"{f}_inep"].isna())
        for f in features_atu
    ],
    axis=1,
).all(axis=1)

linhas_coincidentes = candidatos[coincide].groupby("id_municipio")["linha"].apply(list)
print(f"Municípios da Gold com alguma linha coincidente: {len(linhas_coincidentes)} de {len(g)}")
print(f"Municípios sem nenhuma linha coincidente: {len(g) - len(linhas_coincidentes)}")

unica = linhas_coincidentes[linhas_coincidentes.str.len() == 1].str[0]
print(f"\nMunicípios em que exatamente uma linha coincide ({len(unica)}): qual linha?")
print(unica.value_counts().to_string())

Municípios da Gold com alguma linha coincidente: 5232 de 5232
Municípios sem nenhuma linha coincidente: 0

Municípios em que exatamente uma linha coincide (1365): qual linha?
linha
Total / Municipal    1365


## 7. PIB per capita: qual ano o enriquecimento anterior usou

O PIB per capita só existe na Gold para os municípios dela. Para estendê-lo aos municípios ausentes, calculamos PIB ÷ população (IBGE) para vários anos e identificamos qual reproduz os valores da Gold.

In [13]:
pib = pd.read_parquet(RAW / "ibge_pib.parquet")
populacao = pd.read_parquet(RAW / "ibge_populacao.parquet")

per_capita = pib.merge(populacao, on=["ano", "id_municipio"], validate="one_to_one")
per_capita["pib_per_capita"] = per_capita["pib"].astype(float) / per_capita["populacao"].astype(float)
per_capita["id_municipio"] = per_capita["id_municipio"].astype(str).str.zfill(7)

gold_pib = pd.DataFrame({
    "id_municipio": gold["id_municipio"].astype(str).str.zfill(7),
    "pib_gold": gold["ibge_pib_per_capita"].astype(float),
})

for ano, grupo in per_capita.groupby("ano"):
    pares = gold_pib.merge(grupo, on="id_municipio")
    razao = pares["pib_per_capita"] / pares["pib_gold"]
    erro_relativo = (razao - 1).abs()
    print(
        f"{ano}: {len(pares):>5} pares | iguais (±0,1%): {100 * (erro_relativo < 0.001).mean():5.1f}% "
        f"| iguais (±1%): {100 * (erro_relativo < 0.01).mean():5.1f}% | razão mediana: {razao.median():.4f}"
    )

2019:  5232 pares | iguais (±0,1%):   0.0% | iguais (±1%):   0.2% | razão mediana: 0.6262
2020:  5232 pares | iguais (±0,1%):   0.0% | iguais (±1%):   0.3% | razão mediana: 0.6860
2021:  5232 pares | iguais (±0,1%):   0.1% | iguais (±1%):   1.3% | razão mediana: 0.7754
2022:  5232 pares | iguais (±0,1%):   0.2% | iguais (±1%):   2.6% | razão mediana: 0.8890
2023:  5232 pares | iguais (±0,1%): 100.0% | iguais (±1%): 100.0% | razão mediana: 1.0000


## Conclusões

1. **Alfabetização (Censo 2022):** a `taxa_alfabetizacao` da tabela `municipio` está na escala 0–1; multiplicada por 100, coincide com a taxa recalculada a partir da tabela detalhada, que só contém faixas a partir de 15 anos. Definição confirmada: **população de 15 anos ou mais**. A tabela detalhada permite também a taxa de **25 a 44 anos** (geração dos pais das crianças do 2º ano).
2. **Esgotamento sanitário:** hierarquia confirmada — `Rede geral, rede pluvial ou fossa ligada à rede` é exatamente a soma de `Rede geral ou pluvial` e `Fossa séptica ou fossa filtro ligada à rede` (diferença 0 em todos os municípios). Totais excluem o agregado. Features: % com rede/fossa ligada à rede e % com esgotamento precário.
3. **FUNDEB/SIOPE:** valores acumulados ao longo do ano — o **bimestre 6** é o consolidado anual. Indicadores em R$/aluno usam `valor_real`; percentuais, `valor_percentual`. O indicador 61 foi descartado (mediana 0,00 em todos os bimestres).
4. **Alunos avaliados (rede municipal, 2024):** mediana de 114; 25% dos municípios com até 55 alunos. Com 55 alunos e taxa em torno de 63%, o erro-padrão binomial da taxa é de ~6,5 p.p. — a ponderação do treino pelo número de alunos é necessária.
5. **Cobertura da Gold frente à Silver:** a Silver tem resultado da rede municipal em 2024 para 5.452 municípios; a Gold, para 5.232. Os 220 ausentes (63.841 alunos, ~3,5% do total) concentram-se por UF — Acre (100%), Amazonas (23%), Santa Catarina (18%), Rio Grande do Norte (17%), Rio Grande do Sul (8%) — e têm porte semelhante ao dos incluídos. Causa provável: a Gold parte da tabela de metas municipais, e esses municípios não têm meta publicada. **Decisão: recuperá-los**, construindo a base a partir da Silver.
6. **Base dos Dados (`br_inep_indicadores_educacionais.municipio`) — inválida para 2023:** a carga de 2023 tem colunas deslocadas. Nenhuma rede reproduz a Gold; as identidades composicionais são violadas (grupos de adequação docente somam ~500%, faixas de regularidade ~0,9%, rendimento ~120%) e as medianas são implausíveis (ex.: 34,8 horas-aula diárias). Fonte descartada; mantida no código apenas como evidência.
7. **Planilhas oficiais do INEP — fonte íntegra:** identidades composicionais respeitadas em 100% das linhas; valores conferidos contra a planilha bruta. A Gold usou a linha **localização Total / dependência Municipal** (em todos os 1.365 municípios onde só uma linha da planilha coincide com a Gold, ela é Total/Municipal) — a mesma rede do alvo. Com essa linha, o leitor reproduz a Gold em **100% das 37 features** (diferença máxima 0) e cobre os 5.452 municípios da Silver, inclusive os 220 ausentes.
8. **PIB per capita:** a Gold usa PIB 2023 ÷ população 2023 do IBGE (100% de coincidência; razão mediana 1,0000). O mesmo cálculo estende a variável a todos os municípios.
9. **Base final:** 5.450 municípios (5.452 da Silver menos 2 sem indicador), 25 UFs. Roraima não tem resultado da rede municipal na fonte; o Distrito Federal não tem rede municipal. Os 218 municípios sem meta municipal têm `meta_alfabetizacao` e `atingiu_meta` nulos — afeta apenas o enquadramento com metas.